In [145]:
from featuregraph.datasets import arc_agi
import numpy as np
import matplotlib.pyplot as plt

df_load_data = arc_agi("00576224", split="training")

pair_type = 'train'
pair_index = 0

def reconstruct_grid(df, pair_type, pair_index, grid_role):
    mask = (df['pair_type'] == pair_type) & (df['pair_index'] == pair_index) & (df['grid_role'] == grid_role)
    cells = df[mask]

    return cells.pivot(
        index="row",
        columns="column",
        values="color",
    ).to_numpy()


input = reconstruct_grid(df_load_data, pair_type, pair_index, 'input')
input_flip_horizontal = np.fliplr(input)
input_flip_vertical = np.flipud(input)

output = reconstruct_grid(df_load_data, pair_type, pair_index, 'output')

input_height, input_width = input.shape
output_height, output_width = output.shape

number_of_block_rows = output_height // input_height
number_of_block_columns = output_width // input_width

row, column = np.indices(output.shape)
output_cells = np.array([row.ravel(), column.ravel(), output.ravel()]).T
block_coordinates = np.array([row.ravel() // input_height, column.ravel() // input_width, row.ravel() % input_height, column.ravel() % input_width]).T
original_value = input[block_coordinates[:,2], block_coordinates[:,3]]
horizontal_flip_value = input_flip_horizontal[block_coordinates[:,2], block_coordinates[:,3]]
vertical_flip_value = input_flip_vertical[block_coordinates[:,2], block_coordinates[:,3]]
candidate_transformations = np.array([original_value, horizontal_flip_value, vertical_flip_value]).T
number_of_operators = candidate_transformations.shape[1]
cell_matches = output_cells[:,2:3] == candidate_transformations
block_matches = cell_matches.reshape(number_of_block_rows, input_height, number_of_block_columns, input_width, number_of_operators).all(axis=(1,3))

In [153]:
block_matches
match_counts = block_matches.sum(axis=2)
np.all(match_counts == 1)


np.True_

In [150]:
instruction_layout = np.argmax(block_matches, axis=2)
instruction_layout

array([[0, 0, 0],
       [1, 1, 1],
       [0, 0, 0]])